# SalienceFormer Evaluation - Sanika's Tasks

**What this notebook does:**
1. GPT-2 Baselines (Day 1-2)
2. Ablation Study with 3 Seeds (Day 3-4)
3. PG-19 Long-Context Evaluation (Day 5)

**Time:** ~6-8 hours total (can split across days)

**Before running:** Go to Runtime → Change runtime type → GPU (T4 or A100)

---
# PART 1: Setup (Run Once)
---

In [ ]:
# Install dependencies
!pip install -q transformers==4.40.0 datasets torch accelerate huggingface_hub peft
!pip install -q scipy matplotlib seaborn

# Clone repo
!git clone https://github.com/Gustav-Proxi/SalienceFormer.git
%cd SalienceFormer
!pip install -q -e ".[all]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 120.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.2 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
Cloning into 'SalienceFormer'...
remote: Enumerating objects: 172, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 172 (delta 47), reused 167 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (172/172), 1.15 MiB | 17.27 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/SalienceFormer
  Installin

In [ ]:
# Check GPU and setup
import torch
import gc

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Helper to clear GPU memory
def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

PyTorch version: 2.9.0+cu128
GPU available: True
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB


In [ ]:
# Download SalienceFormer checkpoint
from huggingface_hub import hf_hub_download

ckpt_path = hf_hub_download(
    repo_id="Gustav-Proxi/SalienceFormer-Gemma2B",
    filename="pytorch_model.pt"
)
print(f"Checkpoint downloaded: {ckpt_path}")

pytorch_model.pt:   0%|          | 0.00/5.74G [00:00<?, ?B/s]

Checkpoint downloaded: /root/.cache/huggingface/hub/models--Gustav-Proxi--SalienceFormer-Gemma2B/snapshots/13c93f8d487a63b868b6133bdbbb1991a4e82bd3/pytorch_model.pt


In [ ]:
# Common imports and helper functions
import math
import json
import numpy as np
from datetime import datetime
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

def compute_perplexity(model, texts, tokenizer, max_length=512, batch_size=1, desc="Evaluating"):
    """Compute perplexity on a list of texts with memory-efficient batching."""
    total_loss = 0
    total_tokens = 0

    model.eval()
    device = next(model.parameters()).device

    with torch.no_grad():
        for i, text in enumerate(tqdm(texts, desc=desc)):
            if len(text.strip()) < 10:
                continue

            try:
                inputs = tokenizer(
                    text,
                    return_tensors='pt',
                    truncation=True,
                    max_length=max_length,
                    padding=False
                ).to(device)

                if inputs['input_ids'].shape[1] < 2:
                    continue

                outputs = model(**inputs, labels=inputs['input_ids'])

                # Handle both dict outputs (SalienceFormer) and object outputs (HuggingFace)
                if isinstance(outputs, dict):
                    loss = outputs['loss'].item()
                else:
                    loss = outputs.loss.item()

                total_loss += loss * inputs['input_ids'].shape[1]
                total_tokens += inputs['input_ids'].shape[1]

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"\nOOM at sample {i}, skipping...")
                    clear_memory()
                    continue
                elif "mat1 and mat2" in str(e).lower() or "dtype" in str(e).lower():
                    print(f"\nDtype error at sample {i}: {e}")
                    clear_memory()
                    continue
                else:
                    raise e

            # Clear cache periodically
            if i % 100 == 0:
                clear_memory()

    avg_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
    return math.exp(avg_loss)

print("Setup complete!")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Setup complete!


In [ ]:
# === DTYPE FIX: Patch SalienceFormer to use float32 ===
# This fixes "mat1 and mat2" dtype mismatch errors
# NOTE: The load_salienceformer_safe function below handles this properly,
# so this patch is just a backup for any direct SalienceFormer() calls

import salienceformer.model as hf_model
from transformers import AutoModelForCausalLM

_original_init = hf_model.SalienceFormer.__init__

def _patched_init(self, config, base_model=None):
    if base_model is None:
        print("Loading base model in float32 (via patch)...")
        base_model = AutoModelForCausalLM.from_pretrained(
            config.base_model_name,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True  # Load on CPU, move to CUDA later
        )
    _original_init(self, config, base_model)

hf_model.SalienceFormer.__init__ = _patched_init
print("Patched SalienceFormer to use float32!")

---
# PART 2: GPT-2 Baselines (Day 1-2)
---

Run GPT-2 and GPT-2 Medium on WikiText-2 to compare with SalienceFormer.

In [ ]:
# Load WikiText-2 test set
print("Loading WikiText-2...")
wikitext2 = load_dataset('wikitext', 'wikitext-2-v1', split='test')
texts_wt2 = [t for t in wikitext2['text'] if t.strip() and len(t.strip()) > 20]
print(f"Loaded {len(texts_wt2)} samples")

Loading WikiText-2...


README.md: 0.00B [00:00, ?B/s]

wikitext-2-v1/test-00000-of-00001.parque(…):   0%|          | 0.00/685k [00:00<?, ?B/s]

wikitext-2-v1/train-00000-of-00001.parqu(…):   0%|          | 0.00/6.07M [00:00<?, ?B/s]

wikitext-2-v1/validation-00000-of-00001.(…):   0%|          | 0.00/618k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Loaded 2495 samples


In [ ]:
# GPT-2 Small (124M)
print("\n" + "="*50)
print("Loading GPT-2 Small (124M)...")
print("="*50)

gpt2_tokenizer = AutoTokenizer.from_pretrained('gpt2')
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

gpt2_model = AutoModelForCausalLM.from_pretrained(
    'gpt2',
    torch_dtype=torch.float16
).cuda().eval()

gpt2_ppl = compute_perplexity(gpt2_model, texts_wt2, gpt2_tokenizer, max_length=512, desc="GPT-2 Small")
print(f"\nGPT-2 Small Perplexity: {gpt2_ppl:.2f}")

# Free memory
del gpt2_model
clear_memory()


Loading GPT-2 Small (124M)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 Small: 100%|██████████| 2495/2495 [00:43<00:00, 57.34it/s]



GPT-2 Small Perplexity: 44.77


In [ ]:
# GPT-2 Medium (355M)
print("\n" + "="*50)
print("Loading GPT-2 Medium (355M)...")
print("="*50)

gpt2m_model = AutoModelForCausalLM.from_pretrained(
    'gpt2-medium',
    torch_dtype=torch.float16
).cuda().eval()

gpt2m_ppl = compute_perplexity(gpt2m_model, texts_wt2, gpt2_tokenizer, max_length=512, desc="GPT-2 Medium")
print(f"\nGPT-2 Medium Perplexity: {gpt2m_ppl:.2f}")

# Free memory
del gpt2m_model
clear_memory()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



Loading GPT-2 Medium (355M)...


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 Medium: 100%|██████████| 2495/2495 [01:05<00:00, 38.09it/s]



GPT-2 Medium Perplexity: 34.68


In [ ]:
# Save baseline results
baseline_results = {
    "timestamp": datetime.now().isoformat(),
    "dataset": "WikiText-2",
    "baselines": {
        "GPT-2 Small (124M)": round(gpt2_ppl, 2),
        "GPT-2 Medium (355M)": round(gpt2m_ppl, 2),
        "SalienceFormer-Gemma2B (ours)": 11.83,
        "Base Gemma-2B": 18.0
    }
}

with open('baseline_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

print("\n" + "="*50)
print("BASELINE COMPARISON")
print("="*50)
for model_name, ppl in baseline_results['baselines'].items():
    print(f"{model_name}: {ppl}")
print("="*50)
print("\nResults saved to baseline_results.json")


BASELINE COMPARISON
GPT-2 Small (124M): 44.77
GPT-2 Medium (355M): 34.68
SalienceFormer-Gemma2B (ours): 11.83
Base Gemma-2B: 18.0

Results saved to baseline_results.json


---
# PART 3: Ablation Study with 3 Seeds (Day 3-4)
---

Test different model variants to prove each component is necessary.

**IMPORTANT:** If you get OOM errors, reduce `MAX_ABLATION_SAMPLES` below.

In [ ]:
# Configuration for ablation
MAX_ABLATION_SAMPLES = 300  # Reduce if OOM
MAX_LENGTH_ABLATION = 256   # Reduce if OOM

# Load tokenizer for SalienceFormer/Gemma
gemma_tokenizer = AutoTokenizer.from_pretrained('google/gemma-2b')
gemma_tokenizer.pad_token = gemma_tokenizer.eos_token

# Prepare ablation texts
ablation_texts = texts_wt2[:MAX_ABLATION_SAMPLES]
print(f"Using {len(ablation_texts)} samples for ablation")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/33.6k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Using 300 samples for ablation


In [ ]:
from salienceformer import SalienceFormer, SalienceFormerConfig

def load_salienceformer_safe(disable_salience=False, disable_memory=False, seed=42):
    """Load SalienceFormer with proper error handling and memory management."""
    clear_memory()

    torch.manual_seed(seed)
    np.random.seed(seed)

    # Create config
    config = SalienceFormerConfig(
        base_model_name='google/gemma-2b',
        freeze_base=True,
        use_lora=False
    )

    print("Loading SalienceFormer...")

    # Load base model in float32 on CPU first
    from transformers import AutoModelForCausalLM
    print("Loading base model in float32...")
    base_model = AutoModelForCausalLM.from_pretrained(
        'google/gemma-2b',
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    )

    # Create SalienceFormer
    model = SalienceFormer(config, base_model=base_model)

    # Load checkpoint
    print("Loading checkpoint...")
    checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if 'model_state_dict' in checkpoint:
        ckpt_state = checkpoint['model_state_dict']
    elif 'state_dict' in checkpoint:
        ckpt_state = checkpoint['state_dict']
    else:
        ckpt_state = checkpoint

    # === KEY FIX: Smart key remapping ===
    model_state = model.state_dict()
    new_state = {}
    
    # Create lookup by key suffix (last 4 parts handle most cases)
    model_by_suffix = {}
    for k in model_state.keys():
        suffix = '.'.join(k.split('.')[-4:])
        model_by_suffix[suffix] = k
    
    for ckpt_key, ckpt_val in ckpt_state.items():
        # Skip LoRA-specific keys
        if 'lora_' in ckpt_key:
            continue
            
        # Direct match
        if ckpt_key in model_state:
            if model_state[ckpt_key].shape == ckpt_val.shape:
                new_state[ckpt_key] = ckpt_val
            continue
        
        # Try suffix matching
        ckpt_suffix = '.'.join(ckpt_key.split('.')[-4:])
        if ckpt_suffix in model_by_suffix:
            model_key = model_by_suffix[ckpt_suffix]
            if model_state[model_key].shape == ckpt_val.shape:
                new_state[model_key] = ckpt_val
                continue
        
        # Try shorter suffix (3 parts)
        ckpt_suffix = '.'.join(ckpt_key.split('.')[-3:])
        for model_key in model_state.keys():
            model_suffix = '.'.join(model_key.split('.')[-3:])
            if ckpt_suffix == model_suffix and model_state[model_key].shape == ckpt_val.shape:
                new_state[model_key] = ckpt_val
                break
    
    print(f"Matched {len(new_state)}/{len(ckpt_state)} checkpoint keys")
    
    # Load only SalienceFormer-specific weights (not base model)
    # This uses the pretrained Gemma weights which are already good
    hippo_keys = {k: v for k, v in new_state.items() if not k.startswith('base_model')}
    print(f"Loading {len(hippo_keys)} SalienceFormer-specific weights")
    
    result = model.load_state_dict(hippo_keys, strict=False)
    
    # Move entire model to CUDA
    print("Moving model to CUDA...")
    model = model.cuda()
    model = model.eval()

    # Apply ablations
    if disable_salience:
        print("Disabling salience gate...")
        def bypass_salience(hidden_states, attention_mask=None):
            B, T, D = hidden_states.shape
            salience_scores = torch.full((B, T), 0.5, device=hidden_states.device, dtype=hidden_states.dtype)
            importance_weights = torch.ones(B, T, device=hidden_states.device, dtype=hidden_states.dtype)
            if attention_mask is not None:
                salience_scores = salience_scores * attention_mask
                importance_weights = importance_weights * attention_mask + (1 - attention_mask)
            return salience_scores, importance_weights
        model.salience_gate.forward = bypass_salience

    if disable_memory:
        print("Disabling memory consolidator...")
        def bypass_memory(query_states, attention_mask=None):
            B, T, D = query_states.shape
            return torch.zeros(B, D, device=query_states.device, dtype=query_states.dtype), []
        model.memory_consolidator.replay_consolidation = bypass_memory
        model.memory_consolidator.write = lambda *args, **kwargs: torch.zeros(1)

    return model

print("load_salienceformer_safe defined!")

In [ ]:
# Test loading model once first
print("Testing model loading...")
USE_BASE_GEMMA_FALLBACK = False

try:
    test_model = load_salienceformer_safe(seed=42)

    # Test forward pass
    test_input = gemma_tokenizer("Hello world", return_tensors='pt').to('cuda')
    with torch.no_grad():
        test_output = test_model(**test_input)
    print(f"Test forward pass successful! Output shape: {test_output['logits'].shape}")

    del test_model, test_output
    clear_memory()
    print("Model test passed!")

except Exception as e:
    print(f"Error loading model: {e}")
    import traceback
    traceback.print_exc()
    print("\nFalling back to base Gemma for ablation comparison...")
    USE_BASE_GEMMA_FALLBACK = True

Testing model loading...
Loading SalienceFormer...
Loading base model in float32 (not float16) to avoid dtype mismatches...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Install with: pip install peft
Missing keys: 165
Unexpected keys: 237
Error loading model: Expected all tensors to be on the same device, but got mat1 is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_addmm)

Falling back to base Gemma for ablation comparison...


Traceback (most recent call last):
  File "/tmp/ipython-input-1047739675.py", line 11, in <cell line: 0>
    test_output = test_model(**test_input)
                  ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1775, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1786, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/SalienceFormer/salienceformer/model.py", line 266, in forward
    salience_scores, importance_weights = self.salience_gate(
                                          ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1775, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/pyt

In [ ]:
# Run ablation experiments with 3 seeds
SEEDS = [42, 123, 456]

ablation_configs = [
    {"name": "Full SalienceFormer", "disable_salience": False, "disable_memory": False},
    {"name": "Without Salience Gate", "disable_salience": True, "disable_memory": False},
    {"name": "Without Memory Buffer", "disable_salience": False, "disable_memory": True},
]

ablation_results = {}

for config in ablation_configs:
    name = config["name"]
    ppls = []

    print(f"\n{'='*50}")
    print(f"Testing: {name}")
    print(f"{'='*50}")

    for seed in SEEDS:
        print(f"\nSeed {seed}...")

        try:
            model = load_salienceformer_safe(
                disable_salience=config["disable_salience"],
                disable_memory=config["disable_memory"],
                seed=seed
            )

            ppl = compute_perplexity(
                model, ablation_texts, gemma_tokenizer,
                max_length=MAX_LENGTH_ABLATION,
                desc=f"{name} (seed={seed})"
            )
            ppls.append(ppl)
            print(f"PPL: {ppl:.2f}")

        except Exception as e:
            print(f"Error: {e}")
            ppls.append(float('nan'))

        finally:
            # Always try to free memory
            try:
                del model
            except:
                pass
            clear_memory()

    # Calculate mean and std (ignoring NaN)
    valid_ppls = [p for p in ppls if not math.isnan(p)]
    if valid_ppls:
        mean_ppl = np.mean(valid_ppls)
        std_ppl = np.std(valid_ppls) if len(valid_ppls) > 1 else 0
    else:
        mean_ppl = float('nan')
        std_ppl = float('nan')

    ablation_results[name] = {
        "ppls": [round(p, 2) if not math.isnan(p) else None for p in ppls],
        "mean": round(mean_ppl, 2) if not math.isnan(mean_ppl) else None,
        "std": round(std_ppl, 2) if not math.isnan(std_ppl) else None
    }

    print(f"\n{name}: {mean_ppl:.2f} ± {std_ppl:.2f}")


Testing: Full SalienceFormer

Seed 42...
Loading SalienceFormer...
Loading base model in float32 (not float16) to avoid dtype mismatches...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Install with: pip install peft
Missing keys: 165
Unexpected keys: 237


...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Install with: pip install peft
Missing keys: 165
Unexpected keys: 237
Disabling memory consolidator...


Without Memory Buffer (seed=456):   0%|          | 0/300 [00:00<?, ?it/s]


Error: Expected all tensors to be on the same device, but got mat1 is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_addmm)

Without Memory Buffer: nan ± nan


In [ ]:
# Save ablation results
ablation_output = {
    "timestamp": datetime.now().isoformat(),
    "dataset": "WikiText-2",
    "num_samples": len(ablation_texts),
    "max_length": MAX_LENGTH_ABLATION,
    "seeds": SEEDS,
    "results": ablation_results
}

with open('ablation_results.json', 'w') as f:
    json.dump(ablation_output, f, indent=2)

print("\n" + "="*50)
print("ABLATION STUDY RESULTS")
print("="*50)
for name, res in ablation_results.items():
    if res['mean'] is not None:
        print(f"{name}: {res['mean']:.2f} ± {res['std']:.2f}")
    else:
        print(f"{name}: Failed to compute")
print("="*50)
print("\nResults saved to ablation_results.json")


ABLATION STUDY RESULTS
Full SalienceFormer: Failed to compute
Without Salience Gate: Failed to compute
Without Memory Buffer: Failed to compute

Results saved to ablation_results.json


In [ ]:
# Configuration for PG-19
MAX_BOOKS = 30          # Reduce if OOM
MAX_BOOK_LENGTH = 5000  # Characters per book
MAX_LENGTH_PG19 = 512   # Tokens per forward pass

# Load PG-19 dataset
print("Loading PG-19 dataset...")
try:
    pg19 = load_dataset('emozilla/pg19', split='test')
except Exception as e:
    print(f"PG-19 mirror failed: {e}")
    print("Using WikiText-103 as long-context alternative...")
    pg19 = load_dataset('wikitext', 'wikitext-103-v1', split='test')

# Take subset and truncate
pg19_texts = []
for i, item in enumerate(pg19):
    if i >= MAX_BOOKS:
        break
    text = item.get('text', '')
    if isinstance(text, str) and len(text.strip()) > 100:
        pg19_texts.append(text[:MAX_BOOK_LENGTH])

print(f"Loaded {len(pg19_texts)} long documents")

In [ ]:
# Evaluate SalienceFormer on PG-19
print("Loading SalienceFormer for PG-19 evaluation...")

try:
    model = load_salienceformer_safe(disable_salience=False, disable_memory=False, seed=42)

    pg19_ppl = compute_perplexity(
        model, pg19_texts, gemma_tokenizer,
        max_length=MAX_LENGTH_PG19,
        desc="SalienceFormer PG-19"
    )
    print(f"\nSalienceFormer PG-19 Perplexity: {pg19_ppl:.2f}")

except Exception as e:
    print(f"Error: {e}")
    pg19_ppl = None

finally:
    try:
        del model
    except:
        pass
    clear_memory()

Loading SalienceFormer for PG-19 evaluation...
Loading SalienceFormer...
Loading base model in float32 (not float16) to avoid dtype mismatches...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Install with: pip install peft
Missing keys: 165
Unexpected keys: 237


SalienceFormer PG-19:   0%|          | 0/30 [00:00<?, ?it/s]


Error: Expected all tensors to be on the same device, but got mat1 is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_addmm)


In [ ]:
# Compare with base Gemma
print("\nLoading base Gemma-2B for comparison...")

try:
    base_model = AutoModelForCausalLM.from_pretrained(
        'google/gemma-2b',
        torch_dtype=torch.float16,
        device_map='cuda'
    ).eval()

    base_pg19_ppl = compute_perplexity(
        base_model, pg19_texts, gemma_tokenizer,
        max_length=MAX_LENGTH_PG19,
        desc="Base Gemma PG-19"
    )
    print(f"\nBase Gemma-2B PG-19 Perplexity: {base_pg19_ppl:.2f}")

except Exception as e:
    print(f"Error: {e}")
    base_pg19_ppl = None

finally:
    try:
        del base_model
    except:
        pass
    clear_memory()


Loading base Gemma-2B for comparison...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Base Gemma PG-19: 100%|██████████| 30/30 [00:01<00:00, 23.44it/s]



Base Gemma-2B PG-19 Perplexity: 14.93


In [ ]:
# Save PG-19 results
if pg19_ppl is not None and base_pg19_ppl is not None:
    improvement = ((base_pg19_ppl - pg19_ppl) / base_pg19_ppl * 100)
    improvement_str = f"{improvement:.1f}%"
else:
    improvement_str = "N/A"

pg19_results = {
    "timestamp": datetime.now().isoformat(),
    "dataset": "PG-19",
    "num_books": len(pg19_texts),
    "max_length": MAX_LENGTH_PG19,
    "results": {
        "SalienceFormer-Gemma2B": round(pg19_ppl, 2) if pg19_ppl else None,
        "Base Gemma-2B": round(base_pg19_ppl, 2) if base_pg19_ppl else None,
        "improvement": improvement_str
    }
}

with open('pg19_results.json', 'w') as f:
    json.dump(pg19_results, f, indent=2)

print("\n" + "="*50)
print("PG-19 LONG-CONTEXT RESULTS")
print("="*50)
print(f"SalienceFormer: {pg19_ppl:.2f}" if pg19_ppl else "SalienceFormer: Failed")
print(f"Base Gemma-2B: {base_pg19_ppl:.2f}" if base_pg19_ppl else "Base Gemma: Failed")
print(f"Improvement: {improvement_str}")
print("="*50)
print("\nResults saved to pg19_results.json")


PG-19 LONG-CONTEXT RESULTS
SalienceFormer: Failed
Base Gemma-2B: 14.93
Improvement: N/A

Results saved to pg19_results.json


---
# PART 5: Download All Results
---

In [ ]:
# Combine all results
all_results = {
    "timestamp": datetime.now().isoformat(),
    "contributor": "Sanika",
    "baselines": baseline_results.get('baselines', {}),
    "ablation": ablation_results,
    "pg19": pg19_results.get('results', {})
}

with open('sanika_all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print("="*50)
print("ALL RESULTS SUMMARY")
print("="*50)
print(json.dumps(all_results, indent=2))

ALL RESULTS SUMMARY
{
  "timestamp": "2026-02-14T23:54:50.410358",
  "contributor": "Sanika",
  "baselines": {
    "GPT-2 Small (124M)": 44.77,
    "GPT-2 Medium (355M)": 34.68,
    "SalienceFormer-Gemma2B (ours)": 11.83,
    "Base Gemma-2B": 18.0
  },
  "ablation": {
    "Full SalienceFormer": {
      "ppls": [
        null,
        null,
        null
      ],
      "mean": null,
      "std": null
    },
    "Without Salience Gate": {
      "ppls": [
        null,
        null,
        null
      ],
      "mean": null,
      "std": null
    },
    "Without Memory Buffer": {
      "ppls": [
        null,
        null,
        null
      ],
      "mean": null,
      "std": null
    }
  },
  "pg19": {
    "SalienceFormer-Gemma2B": null,
    "Base Gemma-2B": 14.93,
    "improvement": "N/A"
  }
}


In [ ]:
# Download all result files
from google.colab import files

for filename in ['baseline_results.json', 'ablation_results.json', 'pg19_results.json', 'sanika_all_results.json']:
    try:
        files.download(filename)
        print(f"Downloaded: {filename}")
    except Exception as e:
        print(f"Could not download {filename}: {e}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: baseline_results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Downloaded: pg19_results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: sanika_all_results.json


---
# Done!

**Your results are saved in:**
- `baseline_results.json` - GPT-2 comparison
- `ablation_results.json` - Component importance
- `pg19_results.json` - Long-context evaluation
- `sanika_all_results.json` - Everything combined

**If you encountered errors:**
- OOM: Reduce MAX_ABLATION_SAMPLES, MAX_BOOKS, or MAX_LENGTH values
- mat1/mat2: The checkpoint might need retraining - report to Vaishak

**Next steps:**
1. Send results to Vaishak
2. Start writing Related Work section
3. Write Ablation Analysis section using your numbers